### 0. Environments

In [1]:
!pip install -q ir_datasets bm25s

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.4 MB/s eta 0:00:00


In [1]:
import os
import json
import bm25s
import ir_datasets
from itertools import islice
from tqdm import tqdm
import Stemmer

In [2]:
# configurations
INDEX_DIR = "./bm25_index"
DOC_IDS_PATH = os.path.join(INDEX_DIR, "doc_ids.jsonl")
os.makedirs(INDEX_DIR, exist_ok=True)

### 1. Dataset

In [4]:
!cp -r /kaggle/input/datasets/hongkimgip/msmarco-passage/msmarco-passage /root/.ir_datasets/msmarco-passage

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [3]:
def preview_iter(title, iterator, fields, n=5):
    print(f"\n========== {title} ==========")

    for i, item in enumerate(islice(iterator, n), start=1):
        print(f"\n--- {title[:-1].title()} {i} ---")
        for field in fields:
            value = getattr(item, field)
            if field == "text":
                value = value[:500]
            print(f"{field}:", value)


def download_and_preview_msmarco(
    corpus_id="msmarco-passage",
    eval_id="msmarco-passage/dev/small",
    n_samples=1,
):
    passages = ir_datasets.load(corpus_id)
    eval_set = ir_datasets.load(eval_id)

    preview_iter(
        "SAMPLE PASSAGES",
        passages.docs_iter(),
        fields=["doc_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QUERIES",
        eval_set.queries_iter(),
        fields=["query_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QRELS",
        eval_set.qrels_iter(),
        fields=["query_id", "doc_id", "relevance"],
        n=n_samples,
    )

    corpus = {}
    queries = {}
    qrels = {}

    print("\n========== LOADING FULL CORPUS ==========")
    for doc in tqdm(passages.docs_iter(), desc="Loading passages"):
        corpus[doc.doc_id] = doc.text

    print(f"Total passages loaded: {len(corpus):,}")

    print("\n========== LOADING QUERIES ==========")
    for query in tqdm(eval_set.queries_iter(), desc="Loading queries"):
        queries[query.query_id] = query.text

    print(f"Total queries loaded: {len(queries):,}")

    print("\n========== LOADING QRELS ==========")
    for qrel in tqdm(eval_set.qrels_iter(), desc="Loading qrels"):
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

    print(f"Total qrels queries loaded: {len(qrels):,}")

    return corpus, queries, qrels


corpus, queries, qrels = download_and_preview_msmarco()


========== SAMPLE PASSAGES ==========

--- Sample Passage 1 ---
doc_id: 0
text: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

========== SAMPLE QUERIES ==========

--- Sample Querie 1 ---
query_id: 1048585
text: what is paula deen's brother

========== SAMPLE QRELS ==========

--- Sample Qrel 1 ---
query_id: 300674
doc_id: 7067032
relevance: 1

========== LOADING FULL CORPUS ==========


Loading passages: 8841823it [00:24, 364720.76it/s]


Total passages loaded: 8,841,823

========== LOADING QUERIES ==========


Loading queries: 6980it [00:00, 610137.80it/s]


Total queries loaded: 6,980

========== LOADING QRELS ==========


Loading qrels: 7437it [00:00, 393335.00it/s]

Total qrels queries loaded: 6,980


### 2. Sparse indexing

In [4]:
doc_ids = list(corpus.keys())
texts = [corpus[doc_id] for doc_id in doc_ids]

# Tokenize corpus
stemmer = Stemmer.Stemmer("english")
corpus_tokens = bm25s.tokenize(texts, stopwords="english", stemmer=stemmer)

# Build BM25 index
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# Save BM25 index
retriever.save(INDEX_DIR)

# Save mapping row_id -> doc_id
with open(DOC_IDS_PATH, "w", encoding="utf-8") as f:
    for row_id, doc_id in enumerate(doc_ids):
        f.write(json.dumps({
            "row_id": row_id,
            "doc_id": doc_id,
        }, ensure_ascii=False) + "\n")

print("Saved BM25 index to:", INDEX_DIR)
print("Num docs:", len(doc_ids))

Split strings:   0%|          | 0/8841823 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/8841823 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/8841823 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/8841823 [00:00<?, ?it/s]

Saved BM25 index to: ./bm25_index
Num docs: 8841823


### 3. Sparse indexing with bm25_pt

In [ ]:
import torch
from bm25_pt import BM25 as BM25PT

BM25PT_INDEX_DIR = "./bm25_pt_index"
BM25PT_INDEX_PATH = os.path.join(BM25PT_INDEX_DIR, "bm25_pt_index.pt")
BM25PT_DOC_IDS_PATH = os.path.join(BM25PT_INDEX_DIR, "doc_ids.jsonl")
os.makedirs(BM25PT_INDEX_DIR, exist_ok=True)

In [ ]:
# Build BM25 index using bm25_pt (PyTorch-based)
retriever_pt = BM25PT()  # uses bert-base-uncased tokenizer by default
retriever_pt.index(texts)

# Save BM25-PT index (torch sparse tensors + metadata)
torch.save({
    "corpus_scores": retriever_pt._corpus_scores,
    "corpus": retriever_pt._corpus,
    "corpus_lengths": retriever_pt._corpus_lengths,
    "average_document_length": retriever_pt._average_document_length,
    "documents_containing_word": retriever_pt._documents_containing_word,
    "word_counts": retriever_pt._word_counts,
    "IDF": retriever_pt._IDF,
    "k1": retriever_pt.k1,
    "b": retriever_pt.b,
    "vocab_size": retriever_pt.vocab_size,
}, BM25PT_INDEX_PATH)

# Save mapping row_id -> doc_id
with open(BM25PT_DOC_IDS_PATH, "w", encoding="utf-8") as f:
    for row_id, doc_id in enumerate(doc_ids):
        f.write(json.dumps({
            "row_id": row_id,
            "doc_id": doc_id,
        }, ensure_ascii=False) + "\n")

print("Saved BM25-PT index to:", BM25PT_INDEX_DIR)
print("Num docs:", len(doc_ids))